In [ ]:
from numba import njit
import numpy as np
import optuna
import polars as pl

full_data = pl.read_parquet("../data/1m/1m.parquet")

In [ ]:
symbol = "ETHUSDT"
start_date = pl.datetime(2020, 1, 1)
end_date = pl.datetime(2020, 12, 31)
PCS = 2
data = (
    full_data.filter(
        (pl.col("symbol") == symbol)
        & (pl.col("open_time") >= start_date)
        & (pl.col("open_time") <= end_date)
    )
    .select("open", "high", "low", "close")
    .to_numpy()
)

In [ ]:
@njit
def mm_core(data, lambda_=0.1, spread_=0.1, skew_=0.0):
    n = data.shape[0]
    cash = 0.0
    inventory = 0.0
    pnl_history = np.zeros(n)
    inventory_history = np.zeros(n)
    var = 0.0
    for i in range(n - 1):
        open_price = data[i, 0]
        high_price = data[i, 1]
        low_price = data[i, 2]
        close_price: float = data[i, 3]
        # Rogers–Satchell volatility
        v = (np.log(high_price / close_price) * np.log(high_price / open_price)) + (
            np.log(low_price / close_price) * np.log(low_price / open_price)
        )
        var = lambda_ * v + (1 - lambda_) * var
        # spread = 2 / k * np.log(1 + gamma / k) + gamma * var
        spread = spread_ * np.sqrt(var)
        half_spread = max(spread / 2, 0.01)
        ref_price = close_price - skew_ * inventory
        bid = round(min(ref_price - half_spread, close_price - 10**-PCS), PCS)
        ask = round(max(ref_price + half_spread, close_price + 10**-PCS), PCS)
        if data[i + 1, 2] < bid < data[i + 1, 0]:
            inventory += 1.0
            cash -= bid
        if data[i + 1, 0] < ask < data[i + 1, 1]:
            inventory -= 1.0
            cash += ask
        pnl = cash + inventory * close_price
        pnl_history[i] = pnl
        inventory_history[i] = inventory
    pnl_history[-1] = cash + inventory * data[-1, 3]
    inventory_history[-1] = inventory
    return pnl_history, inventory_history


# @njit
def mm_optimize(data, lambda_=0.1, spread_=0.1, skew_=0.0):
    # todo
    return 0

In [ ]:
study = optuna.create_study()


def objective(trial: optuna.Trial):
    # gamma = trial.suggest_float("gamma", 0.01, 10000.0,log=True)
    lambda_ = trial.suggest_float("lambda_", 0.01, 0.99)
    spread_ = trial.suggest_float("spread_", 1, 1000000000.0, log=True)
    skew_ = trial.suggest_float("skew_", 0.0, 10.0)
    sharpe = mm_optimize(data, lambda_=lambda_, spread_=spread_, skew_=skew_)
    return -sharpe


study.optimize(objective, n_trials=500)

In [ ]:
study.best_params

In [ ]:
rets = mm_core(
    data=data,
    spread_=study.best_params["spread_"],
    lambda_=study.best_params["lambda_"],
    skew_=study.best_params["skew_"],
)
pl.from_numpy(rets[0]).with_columns(
    pl.col("column_0").alias("cum_pnl"),
    pl.arange(0, len(rets[0])).alias("time"),
).group_by_dynamic("time", every="1440i").agg(pl.col("cum_pnl").last()).plot.line(
    x="time", y="cum_pnl"
)


In [ ]:
pl.from_numpy(rets[1]).with_columns(
    pl.col("column_0").alias("inventory"),
    pl.arange(0, len(rets[1]), step=1).alias("time"),
).group_by_dynamic("time", every="1440i").agg(pl.col("inventory").last()).plot.line(
    x="time", y="inventory"
)
